**Week 14 — Cross-validation (validation set, LOOCV, K-fold, Stratified K-fold, one-standard-error rule) บนข้อมูลลูกค้าร้านค้า**

### สรุปเงื่อนไขและข้อตกลงร่วมของ Week 14
- **การอ่านไฟล์:** `cust = pd.read_csv('biz_customers.csv')` (ทั้งหมด 1203 แถว ข้อมูลลูกค้าหนึ่งคนต่อแถว)
- **การแบ่ง Fold (Cross-Validation):** ใช้ลำดับแถวเดิมของไฟล์เสมอ โดย **ไม่สับข้อมูล** (`shuffle=False` ใน `KFold`, `StratifiedKFold`, `LeaveOneOut`; ห้ามใช้ `train_test_split` หรือ `random_state`)
- **การ Standardize:** ในข้อที่ระบุให้ standardize (เช่น KNN) **ต้อง standardize ใหม่ภายในแต่ละ fold โดยคำนวณค่าเฉลี่ยและ SD ($ddof=0$) จาก train fold นั้นเท่านั้น** (ป้องกัน Data Leakage)
- **โมเดลพหุนาม (Polynomial Regression):** ใช้ `PolynomialFeatures(degree=d)` + `LinearRegression()` บนค่าตัวแปรดิบ (ไม่ standardize)
- **One-Standard-Error Rule (1-SE Rule):** เลือกโมเดลที่เรียบง่ายที่สุด (ดีกรีต่ำสุด) ที่มี $\text{CV MSE} \le \text{CV MSE}_{\min} + \text{SE}_{\min}$ โดย $\text{SE} = \frac{\text{SD}_{\text{folds}}(ddof=1)}{\sqrt{K}}$
- **การปัดเศษ:** ทุกคำตอบเป็นตัวเลขทศนิยม 4 ตำแหน่ง (`{ans:.4f}`)

### รายละเอียดไฟล์ข้อมูล `biz_customers.csv` (1203 แถว)

| คอลัมน์ | ความหมายในเรื่องเล่าธุรกิจ | หน่วย / รูปแบบ |
| :--- | :--- | :--- |
| `age` | อายุของลูกค้า | ปี |
| `income_k` | รายได้ของลูกค้า | พันบาท / เดือน |
| `tenure_months` | ระยะเวลาที่เป็นลูกค้า | เดือน |
| `loyalty_member` | การเป็นสมาชิกสะสมแต้ม | 1 = เป็น, 0 = ไม่เป็น |
| `web_visits` | จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด | ครั้ง |
| `email_opens` | จำนวนอีเมลโปรโมชันที่เปิดอ่าน | ฉบับ (จาก 20 ฉบับ) |
| `basket_size` | จำนวนสินค้าเฉลี่ยต่อออเดอร์ | ชิ้น |
| `spend_k` | ยอดซื้อต่อเดือน | พันบาท |
| `churn` | การเลิกซื้อ | 1 = เลิกซื้อ, 0 = ยังซื้อต่อเนื่อง |
| `responded` | การตอบรับโปรโมชัน | 1 = ตอบรับ, 0 = ไม่ตอบรับ |

### 14.1 ทำนาย `responded` ด้วย KNN ($K=11$) และ Stratified 5-Fold Cross-Validation
**โจทย์:** ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ทำนาย `responded` จาก `email_opens`, `income_k`, `basket_size`, `spend_k` ด้วย KNN ที่ $K=11$ โดย standardize ภายในแต่ละ fold ด้วย train fold เท่านั้น ประเมินด้วย Stratified 5-Fold CV (`StratifiedKFold(n_splits=5, shuffle=False)`) หาค่าเฉลี่ยของ F1-score (คลาสบวก = 1) ของทั้ง 5 validation folds (ทศนิยม 4 ตำแหน่ง)

$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \text{TP}}{2 \text{TP} + \text{FP} + \text{FN}}$$
$$\text{CV F1} = \frac{1}{5} \sum_{k=1}^5 \text{F1}_k$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

X = cust[['email_opens', 'income_k', 'basket_size', 'spend_k']].values
y = cust['responded'].values

skf = StratifiedKFold(n_splits=5, shuffle=False)
f1_scores = []

for train_idx, val_idx in skf.split(X, y):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X[train_idx])
    X_val_scaled = scaler.transform(X[val_idx])
    
    knn = KNeighborsClassifier(n_neighbors=11)
    knn.fit(X_train_scaled, y[train_idx])
    y_pred = knn.predict(X_val_scaled)
    
    f1_scores.append(f1_score(y[val_idx], y_pred, pos_label=1))

ans_14_1 = np.mean(f1_scores)
print(f"F1 scores per fold: {[round(s, 4) for s in f1_scores]}")
print(f"คำตอบ 14.1 (CV F1 Score): {ans_14_1:.4f}")


F1 scores per fold: [0.2626, 0.283, 0.1754, 0.2991, 0.2524]
คำตอบ 14.1 (CV F1 Score): 0.2545


### 14.2 การแบ่ง Fold ด้วย K-Fold ($K=7$) และค่าเฉลี่ยของ Validation Fold ที่ 7
**โจทย์:** แบ่งลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ด้วย `KFold(n_splits=7, shuffle=False)` ตามลำดับแถวเดิม หาค่าเฉลี่ยของ `income_k` ของลูกค้าที่อยู่ใน validation fold ลำดับที่ 7 (นับ fold แรกเป็นลำดับที่ 1) (ทศนิยม 4 ตำแหน่ง)

$$\bar{x}_{\text{fold 7}} = \frac{1}{n_7} \sum_{i \in \text{Fold}_7} \text{income\_k}_i$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

kf = KFold(n_splits=7, shuffle=False)
splits = list(kf.split(cust))

# Fold ลำดับที่ 7 (index 6 ใน 0-based)
_, val_idx_7 = splits[6]
ans_14_2 = cust.iloc[val_idx_7]['income_k'].mean()

print(f"จำนวนลูกค้าใน Validation Fold 7: {len(val_idx_7)} คน (แถวที่ {val_idx_7[0]} ถึง {val_idx_7[-1]})")
print(f"คำตอบ 14.2 (Mean income_k ใน Fold 7): {ans_14_2:.4f}")


จำนวนลูกค้าใน Validation Fold 7: 171 คน (แถวที่ 1032 ถึง 1202)
คำตอบ 14.2 (Mean income_k ใน Fold 7): 28.6515


### 14.3 ประเมิน Linear Regression ด้วย K-Fold ($K=5$) Cross-Validation
**โจทย์:** ทำนายยอดซื้อต่อเดือน (`spend_k`) จากระยะเวลาที่เป็นลูกค้า (`tenure_months`), อายุ (`age`), และอีเมลที่เปิดอ่าน (`email_opens`) ด้วย Linear Regression โดยใช้ลูกค้าทั้ง 1203 คน ประเมินด้วย `KFold(n_splits=5, shuffle=False)` หาค่าเฉลี่ยของ validation MSE ของทั้ง 5 fold (CV MSE) (ทศนิยม 4 ตำแหน่ง)

$$\text{MSE}_k = \frac{1}{|\mathcal{V}_k|} \sum_{i \in \mathcal{V}_k} (y_i - \hat{y}_i)^2$$
$$\text{CV MSE} = \frac{1}{K} \sum_{k=1}^K \text{MSE}_k$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

X = cust[['tenure_months', 'age', 'email_opens']].values
y = cust['spend_k'].values

kf = KFold(n_splits=5, shuffle=False)
val_mses = []

for train_idx, val_idx in kf.split(X):
    lr = LinearRegression()
    lr.fit(X[train_idx], y[train_idx])
    y_pred = lr.predict(X[val_idx])
    val_mses.append(mean_squared_error(y[val_idx], y_pred))

ans_14_3 = np.mean(val_mses)
print(f"Validation MSEs per fold: {[round(m, 4) for m in val_mses]}")
print(f"คำตอบ 14.3 (CV MSE): {ans_14_3:.4f}")


Validation MSEs per fold: [1.9577, 2.2805, 2.5477, 2.8756, 2.3223]
คำตอบ 14.3 (CV MSE): 2.3968


### 14.4 Validation Set Approach (แบ่งแถวคู่-คี่) สำหรับโมเดลพหุนามดีกรี 2
**โจทย์:** แบ่งลูกค้า 1203 คนแบบไม่สุ่ม: แถวที่เป็นเลขคู่ ($0, 2, 4, \dots$ รวม 602 คน) เป็น Training set และแถวที่เป็นเลขคี่ ($1, 3, 5, \dots$ รวม 601 คน) เป็น Validation set ฝึกโมเดลพหุนามดีกรี 2 ทำนายจำนวนสินค้าเฉลี่ยต่อออเดอร์ (`basket_size`) จากอายุของลูกค้า (`age`) บน Training set แล้วคำนวณ MSE บน Validation set (ทศนิยม 4 ตำแหน่ง)

$$\hat{y} = \hat{\beta}_0 + \hat{\beta}_1 \text{age} + \hat{\beta}_2 \text{age}^2$$
$$\text{MSE}_{\text{val}} = \frac{1}{n_{\text{val}}} \sum_{i \in \text{val}} (y_i - \hat{y}_i)^2$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

# แบ่งชุดข้อมูลแถวคู่ (train) และแถวคี่ (val)
train_idx = np.arange(0, len(cust), 2)
val_idx = np.arange(1, len(cust), 2)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(cust[['age']].values)
y = cust['basket_size'].values

lr = LinearRegression()
lr.fit(X_poly[train_idx], y[train_idx])
y_val_pred = lr.predict(X_poly[val_idx])

ans_14_4 = mean_squared_error(y[val_idx], y_val_pred)
print(f"จำนวน Train (แถวคู่): {len(train_idx)}, Validation (แถวคี่): {len(val_idx)}")
print(f"คำตอบ 14.4 (Validation MSE): {ans_14_4:.4f}")


จำนวน Train (แถวคู่): 602, Validation (แถวคี่): 601
คำตอบ 14.4 (Validation MSE): 0.6928


### 14.5 Leave-One-Out Cross-Validation (LOOCV) บนกลุ่มลูกค้า 150 คนแรก
**โจทย์:** ใช้ลูกค้ากลุ่มนำร่อง 150 คนแรก (แถวที่ 0 ถึง 149) ประเมินโมเดล KNN ($K=9$) ทำนายการตอบรับโปรโมชัน (`responded`) จากรายได้ (`income_k`), ระยะเวลาที่เป็นลูกค้า (`tenure_months`), และอายุ (`age`) โดย standardize ข้อมูลใหม่ในแต่ละรอบด้วย train fold (149 คน) เท่านั้น ด้วย Leave-One-Out CV (`LeaveOneOut()`) หาสัดส่วนลูกค้าที่ทำนายผิดพลาด (Classification Error Rate) (ทศนิยม 4 ตำแหน่ง)

$$\text{Error Rate} = \frac{1}{n} \sum_{i=1}^n \mathbb{I}(y_i \ne \hat{y}_{(-i)})$$เมื่อ $\hat{y}_{(-i)}$ คือผลทำนายจุดที่ $i$ จากโมเดลที่ fit โดยตัดจุดที่ $i$ ออก

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

# กลุ่มนำร่อง 150 คนแรก
pilot = cust.iloc[:150].copy()
X = pilot[['income_k', 'tenure_months', 'age']].values
y = pilot['responded'].values

loo = LeaveOneOut()
errors = []

for train_idx, val_idx in loo.split(X):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X[train_idx])
    X_val_scaled = scaler.transform(X[val_idx])
    
    knn = KNeighborsClassifier(n_neighbors=9)
    knn.fit(X_train_scaled, y[train_idx])
    pred = knn.predict(X_val_scaled)[0]
    
    errors.append(int(pred != y[val_idx][0]))

ans_14_5 = np.mean(errors)
print(f"จำนวนครั้งที่ทำนายผิด: {sum(errors)} จาก {len(errors)} คน")
print(f"คำตอบ 14.5 (LOOCV Error Rate): {ans_14_5:.4f}")


จำนวนครั้งที่ทำนายผิด: 43 จาก 150 คน
คำตอบ 14.5 (LOOCV Error Rate): 0.2867


### 14.6 การเลือกดีกรีพหุนามด้วยกฎ One-Standard-Error Rule (1-SE Rule)
**โจทย์:** ทำนายยอดซื้อต่อเดือน (`spend_k`) จากอายุ (`age`) โดยทดลองพหุนามดีกรี 1 ถึง 3 ประเมินแต่ละดีกรีด้วย `KFold(n_splits=5, shuffle=False)` และใช้กฎ One-Standard-Error Rule:
1. หาดีกรีที่มี CV MSE ต่ำสุด และคำนวณ Standard Error ของดีกรีนั้น: $\text{SE} = \frac{\text{SD}(ddof=1)}{\sqrt{5}}$
2. กำหนดเกณฑ์ $\text{Threshold} = \text{CV MSE}_{\min} + \text{SE}$
3. เลือกดีกรีที่ **ต่ำที่สุด** ที่มีค่า $\text{CV MSE} \le \text{Threshold}$
หาค่า CV MSE ของดีกรีที่ได้รับเลือก (ทศนิยม 4 ตำแหน่ง)

$$\text{SE}_{\min} = \frac{1}{\sqrt{K}} \sqrt{\frac{1}{K-1} \sum_{k=1}^K (\text{MSE}_k - \overline{\text{MSE}})^2}$$
$$d^* = \min \{ d \in \{1, 2, 3\} \mid \text{CV MSE}(d) \le \text{CV MSE}_{\min} + \text{SE}_{\min} \}$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

kf = KFold(n_splits=5, shuffle=False)
deg_cv_mse = {}
deg_fold_mses = {}

for d in [1, 2, 3]:
    poly = PolynomialFeatures(degree=d, include_bias=False)
    X_poly = poly.fit_transform(cust[['age']].values)
    y = cust['spend_k'].values
    mses = []
    for train_idx, val_idx in kf.split(X_poly):
        lr = LinearRegression()
        lr.fit(X_poly[train_idx], y[train_idx])
        mses.append(mean_squared_error(y[val_idx], lr.predict(X_poly[val_idx])))
    deg_cv_mse[d] = np.mean(mses)
    deg_fold_mses[d] = mses
    print(f"ดีกรี {d}: CV MSE = {deg_cv_mse[d]:.4f}, MSE แต่ละ fold: {[round(m, 4) for m in mses]}")

# หาดีกรีที่มี CV MSE ต่ำสุด
best_deg = min(deg_cv_mse, key=deg_cv_mse.get)
min_cv_mse = deg_cv_mse[best_deg]
se = np.std(deg_fold_mses[best_deg], ddof=1) / np.sqrt(5)
threshold = min_cv_mse + se

print(f"\nดีกรีที่มี CV MSE ต่ำสุด: ดีกรี {best_deg} (CV MSE = {min_cv_mse:.4f})")
print(f"SE ของดีกรี {best_deg}: {se:.4f}")
print(f"เกณฑ์ Threshold (min + SE): {threshold:.4f}")

# เลือกดีกรีต่ำสุดที่ผ่านเกณฑ์
chosen_deg = min([d for d in [1, 2, 3] if deg_cv_mse[d] <= threshold])
ans_14_6 = deg_cv_mse[chosen_deg]

print(f"ดีกรีต่ำสุดที่เลือกตามกฎ 1-SE: ดีกรี {chosen_deg}")
print(f"คำตอบ 14.6 (CV MSE ของดีกรีที่ถูกเลือก): {ans_14_6:.4f}")


ดีกรี 1: CV MSE = 2.4376, MSE แต่ละ fold: [2.0537, 2.4138, 2.57, 2.8309, 2.3198]
ดีกรี 2: CV MSE = 2.3869, MSE แต่ละ fold: [1.9986, 2.4117, 2.5471, 2.7806, 2.1967]
ดีกรี 3: CV MSE = 2.3804, MSE แต่ละ fold: [2.0057, 2.4194, 2.5252, 2.7772, 2.1746]

ดีกรีที่มี CV MSE ต่ำสุด: ดีกรี 3 (CV MSE = 2.3804)
SE ของดีกรี 3: 0.1347
เกณฑ์ Threshold (min + SE): 2.5152
ดีกรีต่ำสุดที่เลือกตามกฎ 1-SE: ดีกรี 1
คำตอบ 14.6 (CV MSE ของดีกรีที่ถูกเลือก): 2.4376


### 14.7 ผลต่างระหว่าง Validation MSE เฉลี่ย กับ Train MSE เฉลี่ย (โมเดลพหุนามดีกรี 3)
**โจทย์:** ฝึกโมเดลพหุนามดีกรี 3 ทำนาย `spend_k` จาก `age` ใช้ลูกค้าทั้ง 1203 คน ประเมินด้วย `KFold(n_splits=5, shuffle=False)` หาค่าเฉลี่ยของ MSE บน validation ของทั้ง 5 fold ลบด้วยค่าเฉลี่ยของ MSE บน train fold ของทั้ง 5 fold (ทศนิยม 4 ตำแหน่ง)

$$\Delta_{\text{MSE}} = \overline{\text{MSE}}_{\text{val}} - \overline{\text{MSE}}_{\text{train}} = \left( \frac{1}{5} \sum_{k=1}^5 \text{MSE}_{\text{val}, k} \right) - \left( \frac{1}{5} \sum_{k=1}^5 \text{MSE}_{\text{train}, k} \right)$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(cust[['age']].values)
y = cust['spend_k'].values

kf = KFold(n_splits=5, shuffle=False)
train_mses = []
val_mses = []

for train_idx, val_idx in kf.split(X_poly):
    lr = LinearRegression()
    lr.fit(X_poly[train_idx], y[train_idx])
    train_mses.append(mean_squared_error(y[train_idx], lr.predict(X_poly[train_idx])))
    val_mses.append(mean_squared_error(y[val_idx], lr.predict(X_poly[val_idx])))

mean_val_mse = np.mean(val_mses)
mean_train_mse = np.mean(train_mses)
ans_14_7 = mean_val_mse - mean_train_mse

print(f"Mean Validation MSE: {mean_val_mse:.4f}")
print(f"Mean Train MSE: {mean_train_mse:.4f}")
print(f"คำตอบ 14.7 (Mean Val MSE - Mean Train MSE): {ans_14_7:.4f}")


Mean Validation MSE: 2.3804
Mean Train MSE: 2.3687
คำตอบ 14.7 (Mean Val MSE - Mean Train MSE): 0.0117


### 14.8 ประเมิน LDA ทำนายการเลิกซื้อ (`churn`) ด้วย Stratified 5-Fold CV (AUC)
**โจทย์:** เนื่องจากลูกค้าที่เลิกซื้อ (`churn = 1`) มีสัดส่วนน้อย จึงใช้ Stratified K-Fold เพื่อรักษาสัดส่วนคลาสในแต่ละ fold ใช้ลูกค้าทั้ง 1203 คนฝึก Linear Discriminant Analysis (LDA) ทำนาย `churn` จากระยะเวลาที่เป็นลูกค้า (`tenure_months`), จำนวนสินค้าเฉลี่ย (`basket_size`), และรายได้ (`income_k`) ประเมินด้วย `StratifiedKFold(n_splits=5, shuffle=False)` โดยใช้ `churn` เป็นตัวแบ่งชั้น หาค่าเฉลี่ยของ AUC บน validation fold ทั้ง 5 fold (ใช้ความน่าจะเป็นของคลาส 1) (ทศนิยม 4 ตำแหน่ง)

$$\text{CV AUC} = \frac{1}{5} \sum_{k=1}^5 \text{AUC}_k$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

X = cust[['tenure_months', 'basket_size', 'income_k']].values
y = cust['churn'].values

skf = StratifiedKFold(n_splits=5, shuffle=False)
auc_scores = []

for train_idx, val_idx in skf.split(X, y):
    lda = LinearDiscriminantAnalysis()
    lda.fit(X[train_idx], y[train_idx])
    probs = lda.predict_proba(X[val_idx])[:, 1]
    auc_scores.append(roc_auc_score(y[val_idx], probs))

ans_14_8 = np.mean(auc_scores)
print(f"AUC per fold: {[round(a, 4) for a in auc_scores]}")
print(f"คำตอบ 14.8 (CV AUC): {ans_14_8:.4f}")


AUC per fold: [0.7285, 0.5774, 0.7028, 0.6686, 0.7202]
คำตอบ 14.8 (CV AUC): 0.6795


### 14.9 ทำนาย `basket_size` ด้วย KNN Regression ($K=15$) และ 5-Fold CV
**โจทย์:** ทำนายจำนวนสินค้าเฉลี่ยต่อออเดอร์ (`basket_size`) จากยอดซื้อ (`spend_k`), รายได้ (`income_k`), และระยะเวลาลูกค้า (`tenure_months`) ด้วย KNN Regression ที่ $K=15$ โดย standardize ข้อมูลใหม่ภายในแต่ละ fold ด้วย train fold เท่านั้น ใช้ลูกค้าทั้ง 1203 คน ประเมินด้วย `KFold(n_splits=5, shuffle=False)` หาค่าเฉลี่ยของ validation MSE ของทั้ง 5 fold (CV MSE) (ทศนิยม 4 ตำแหน่ง)

$$\hat{y}(x) = \frac{1}{K} \sum_{i \in \mathcal{N}_K(x)} y_i$$
$$\text{CV MSE} = \frac{1}{5} \sum_{k=1}^5 \text{MSE}_k$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

X = cust[['spend_k', 'income_k', 'tenure_months']].values
y = cust['basket_size'].values

kf = KFold(n_splits=5, shuffle=False)
val_mses = []

for train_idx, val_idx in kf.split(X):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X[train_idx])
    X_val_scaled = scaler.transform(X[val_idx])
    
    knnr = KNeighborsRegressor(n_neighbors=15)
    knnr.fit(X_train_scaled, y[train_idx])
    y_pred = knnr.predict(X_val_scaled)
    
    val_mses.append(mean_squared_error(y[val_idx], y_pred))

ans_14_9 = np.mean(val_mses)
print(f"Validation MSEs per fold: {[round(m, 4) for m in val_mses]}")
print(f"คำตอบ 14.9 (CV MSE): {ans_14_9:.4f}")


Validation MSEs per fold: [0.6446, 0.6665, 0.6426, 0.7202, 0.6112]
คำตอบ 14.9 (CV MSE): 0.6570


### 14.10 เปรียบเทียบผลประเมินระหว่าง Single Train/Test Split (วิธี A) กับ K-Fold CV (วิธี B)
**โจทย์:** ใช้ลูกค้า 1203 คนฝึก Linear Regression ทำนาย `spend_k` จาก `income_k`, `basket_size`, `web_visits` เปรียบเทียบ 2 วิธี:
- **วิธี A (Single Split):** Train = 960 แถวแรก, Test = 243 แถวที่เหลือ หา MSE บน Test set
- **วิธี B (5-Fold CV):** `KFold(n_splits=5, shuffle=False)` บนข้อมูลทั้ง 1203 แถว หาค่าเฉลี่ยของ validation MSE (CV MSE)
คำนวณค่า **(CV MSE ของวิธี B) ลบด้วย (Test MSE ของวิธี A)** (ทศนิยม 4 ตำแหน่ง ค่าติดลบได้)

$$\Delta_{\text{B-A}} = \text{CV MSE}_B - \text{MSE}_{\text{test}, A}$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import os

csv_path = 'biz_customers.csv' if os.path.exists('biz_customers.csv') else 'file_csv/biz_customers.csv'
cust = pd.read_csv(csv_path)

X = cust[['income_k', 'basket_size', 'web_visits']].values
y = cust['spend_k'].values

# วิธี A: แบ่งครั้งเดียว train 960 คนแรก, test 243 คนหลัง
X_train_A, X_test_A = X[:960], X[960:]
y_train_A, y_test_A = y[:960], y[960:]

lr_A = LinearRegression()
lr_A.fit(X_train_A, y_train_A)
mse_A = mean_squared_error(y_test_A, lr_A.predict(X_test_A))

# วิธี B: 5-Fold Cross-Validation บนทั้ง 1203 แถว
kf = KFold(n_splits=5, shuffle=False)
mses_B = []
for train_idx, val_idx in kf.split(X):
    lr_B = LinearRegression()
    lr_B.fit(X[train_idx], y[train_idx])
    mses_B.append(mean_squared_error(y[val_idx], lr_B.predict(X[val_idx])))
cv_mse_B = np.mean(mses_B)

ans_14_10 = cv_mse_B - mse_A
print(f"วิธี A (Test MSE บน 243 แถวหลัง): {mse_A:.4f}")
print(f"วิธี B (5-Fold CV MSE): {cv_mse_B:.4f}")
print(f"คำตอบ 14.10 (CV MSE_B - Test MSE_A): {ans_14_10:.4f}")


วิธี A (Test MSE บน 243 แถวหลัง): 1.7618
วิธี B (5-Fold CV MSE): 1.7789
คำตอบ 14.10 (CV MSE_B - Test MSE_A): 0.0171


## สรุปทฤษฎีและสูตรสำคัญ: Cross-Validation & Model Selection (Week 14)

การประเมินประสิทธิภาพของโมเดลการเรียนรู้ทางสถิติ (Statistical Learning) จำเป็นต้องวัดความสามารถในการทำนายบน **ข้อมูลที่ไม่เคยเห็นในขั้นตอนฝึก (Unseen Data)** เพื่อหลีกเลี่ยงปัญหา Overfitting เทคนิค Cross-Validation (CV) จึงเป็นรากฐานที่สำคัญที่สุดในการวัด Generalization Error และคัดเลือกโมเดล (Model Selection / Hyperparameter Tuning)

---

### 1. การเปรียบเทียบระเบียบวิธี Validation ต่างๆ

| ระเบียบวิธี (Validation Method) | หลักการและข้อดี | ข้อจำกัด / ข้อควรระวัง |
| :--- | :--- | :--- |
| **Validation Set Approach** | แบ่งข้อมูลออกเป็น 2 ส่วน (Train/Validation) แบบคงที่ เช่น 70:30 หรือ แถวคู่/คี่ รวดเร็วและคำนวณง่าย | ผลประเมินมีความแปรปรวนสูง (High Variance) ขึ้นอยู่กับว่าจุดใดตกอยู่ใน Train หรือ Validation และเสียข้อมูลฝึกไปจำนวนมาก |
| **Leave-One-Out CV (LOOCV)** | มี $n$ รอบ โดยแต่ละรอบใช้ $n-1$ ตัวอย่างฝึก และ 1 ตัวอย่างทดสอบ ($K = n$) ได้ผลประมาณที่แทบไม่มี Bias (Almost Unbiased) | ต้นทุนการคำนวณสูงมากเมื่อ $n$ ใหญ่ และผลประเมินในแต่ละรอบ correlated กันสูงทำให้เกิด Variance สูงในโมเดลบางชนิด |
| **$K$-Fold CV** | แบ่งข้อมูลเป็น $K$ ส่วนเท่าๆ กัน (นิยม $K=5$ หรือ $10$) หมุนเวียนแต่ละส่วนเป็น Validation ลดทั้ง Bias และ Variance ได้สมดุล (Bias-Variance Trade-Off ดีที่สุด) | ต้องระวังเรื่องลำดับข้อมูลและการป้องกัน Data Leakage ระหว่าง Fold |
| **Stratified $K$-Fold CV** | เหมือน $K$-Fold แต่คงสัดส่วนของแต่ละคลาสในทุก Fold ให้เท่ากับสัดส่วนในข้อมูลจริง | สำคัญอย่างยิ่งสำหรับปัญหา **Imbalanced Classification** เช่น Churn หรือ Fraud Detection |

---

### 2. สูตรและคณิตศาสตร์ของ Cross-Validation

#### 2.1 $K$-Fold Cross-Validation Error
เมื่อแบ่งข้อมูลออกเป็น $K$ กลุ่มย่อย $\mathcal{C}_1, \mathcal{C}_2, \dots, \mathcal{C}_K$ โดยมีขนาดกลุ่มละ $n_k$ (โดย $\sum n_k = n$):
$$\text{CV}_{(K)} = \frac{1}{K} \sum_{k=1}^K \text{MSE}_k, \quad \text{เมื่อ } \text{MSE}_k = \frac{1}{n_k} \sum_{i \in \mathcal{C}_k} (y_i - \hat{y}_i^{(-k)})^2$$
โดย $\hat{y}_i^{(-k)}$ คือค่าพยากรณ์จากโมเดลที่ fit โดยใช้ข้อมูลทั้งหมดยกเว้นกลุ่ม $\mathcal{C}_k$

#### 2.2 Standard Error ของ CV Error
เพื่อวัดความผันแปรของการประเมินข้าม Fold เราคำนวณ Sample Standard Deviation ($ddof=1$) และ Standard Error (SE):
$$\text{SD} = \sqrt{\frac{1}{K-1} \sum_{k=1}^K (\text{MSE}_k - \text{CV}_{(K)})^2}$$
$$\text{SE}(\text{CV}_{(K)}) = \frac{\text{SD}}{\sqrt{K}}$$

---

### 3. กฎ One-Standard-Error Rule (1-SE Rule)
หลักการ **Parsimony (Occam's Razor)**: หากสองโมเดลให้ประสิทธิภาพใกล้เคียงกัน ให้เลือกโมเดลที่เรียบง่ายกว่า (Simpler Model) เพื่อป้องกัน Overfitting และตีความได้ง่ายขึ้น

1. หาโมเดลที่มีค่าประเมินผิดพลาดต่ำที่สุด $\text{CV MSE}_{\min}$ และคำนวณ $\text{SE}_{\min}$
2. กำหนดเกณฑ์เพดาน (Threshold):
   $$\text{Threshold} = \text{CV MSE}_{\min} + \text{SE}_{\min}$$
3. **เลือกโมเดลที่เรียบง่ายที่สุด** (เช่น Degree ต่ำที่สุด, สัมประสิทธิ์น้อยที่สุด, หรือ Regularization สูงที่สุด) ที่มีค่า $\text{CV MSE} \le \text{Threshold}$

---

### 4. กฎเหล็กป้องกัน Data Leakage ในกระบวนการ Cross-Validation

> [!CAUTION]
> **ห้าม Standardize หรือทำ Feature Preprocessing บนข้อมูลทั้งหมดก่อนแบ่ง Fold เด็ดขาด!**

- หากนำข้อมูลทั้งชุดมาคำนวณ $\mu$ และ $\sigma$ ก่อน จะทำให้ข้อมูลใน Validation Set รั่วไหล (Leak) เข้าไปในโมเดล ส่งผลให้ผลประเมินดูดีเกินจริง (Optimistic Bias)
- **วิธีที่ถูกต้อง:** ภายในแต่ละลูปของ Fold ให้คำนวณสถิติด้วย `scaler.fit_transform(X_train)` จาก Train Fold เท่านั้น แล้วนำ scaler นั้นไปใช้ `scaler.transform(X_val)` บน Validation Fold

---

### 5. สรุปคำตอบสถิติและการคำนวณประจำ Week 14

| ข้อ | หัวข้อ / วิธีการ | โมเดล / ตัวแปร | ตัวชี้วัด | คำตอบ |
| :---: | :--- | :--- | :--- | :---: |
| **14.1** | Stratified 5-Fold CV (Scaled per fold) | KNN ($K=11$) $\rightarrow$ `responded` | CV F1-Score | **0.2545** |
| **14.2** | K-Fold ($K=7$) Fold Structure | ค่าเฉลี่ย `income_k` ใน Val Fold 7 | Mean (พันบาท) | **28.6515** |
| **14.3** | 5-Fold CV Regression | Linear Regression $\rightarrow$ `spend_k` | CV MSE | **2.3968** |
| **14.4** | Validation Set (คู่/คี่) | Poly Degree 2 $\rightarrow$ `basket_size` | Val MSE | **0.6928** |
| **14.5** | Leave-One-Out CV (LOOCV 150 คนแรก) | KNN ($K=9$) $\rightarrow$ `responded` | Error Rate | **0.2867** |
| **14.6** | One-Standard-Error Rule | Poly (1..3) $\rightarrow$ `spend_k` (เลือก Deg 1) | Selected CV MSE | **2.4376** |
| **14.7** | Generalization Gap | Poly Degree 3 $\rightarrow$ `spend_k` | Val MSE - Train MSE | **0.0117** |
| **14.8** | Stratified 5-Fold CV (Imbalance) | LDA $\rightarrow$ `churn` | CV AUC | **0.6795** |
| **14.9** | 5-Fold CV KNN Regression | KNN ($K=15$) $\rightarrow$ `basket_size` | CV MSE | **0.6570** |
| **14.10** | Single Split vs. 5-Fold CV | Linear Regression $\rightarrow$ `spend_k` | $\text{CV MSE}_B - \text{MSE}_A$ | **0.0171** |